# 智慧垃圾分類 — YOLO11 訓練 Notebook

**執行前必做：Runtime → Change runtime type → T4 GPU**

## 執行順序
1. Cell 1：確認 GPU
2. Cell 2：安裝套件
3. Cell 3：Clone 專案
4. Cell 4：下載 TrashNet 資料集（需 Kaggle API Token）
5. Cell 5：TrashNet 格式轉換
6. Cell 6：Clone TACO repo（取得 annotations.json）
7. Cell 6b：從 Zenodo 下載完整 TACO 圖片（2.7GB，官方備份）
8. Cell 6c：TACO 轉換 + 合併 → data/merged_v3
9. Cell 7：訓練（waste_sorter_v3）
10. Cell 8：下載 best_v3.pt

> **Class 4（鋁箔包）**：TACO 含 Drink carton 標註，Zenodo 備份可完整下載。

In [ ]:
# ── Cell 1：確認 GPU ───────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('❌ 沒有 GPU，請先到 Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2：安裝套件 ───────────────────────────────────────────────────────────
!pip install ultralytics kaggle requests Pillow -q
print('套件安裝完成')

In [ ]:
# ── Cell 3：Clone 專案 ─────────────────────────────────────────────────────────
import os

REPO = 'AI-course'
if os.path.exists(REPO):
    print('Repo 已存在，執行 git pull...')
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Saibusu/AI-course.git

%cd /content/AI-course
print('工作目錄：', os.getcwd())

In [ ]:
# ── Cell 4：下載 TrashNet（Kaggle API Token）───────────────────────────────────
# 取得方式：kaggle.com/settings → API → 複製 Token（格式：KGAT_...）
import os

os.environ['KAGGLE_TOKEN'] = 'YOUR_KAGGLE_API_TOKEN'  # ← 貼上你的 Token，勿上傳 GitHub

!kaggle datasets download -d asdasdasasdas/garbage-classification -p data/
!unzip -q data/garbage-classification.zip -d data/TrashNet

# 確認結構
for root, dirs, files in os.walk('data/TrashNet'):
    level = root.replace('data/TrashNet', '').count(os.sep)
    if level == 2:
        print(f'{os.path.basename(root)}/: {len(files)} files')
    if level > 2:
        break

In [ ]:
# ── Cell 5：TrashNet → YOLO 6-class 格式轉換 ──────────────────────────────────
# 已知：Kaggle 資料集實際路徑為兩層子目錄
# data/TrashNet/garbage classification/Garbage classification/{glass,metal,...}/
import os

TRASHNET_DIR = 'data/TrashNet/garbage classification/Garbage classification'

if not os.path.exists(TRASHNET_DIR):
    # 嘗試大寫版本
    TRASHNET_DIR = 'data/TrashNet/Garbage classification/Garbage classification'

print('TrashNet 來源路徑:', TRASHNET_DIR)
print('子目錄：', os.listdir(TRASHNET_DIR))

!python data/prepare_trashnet.py \
    --trashnet-dir "{TRASHNET_DIR}" \
    --output-dir   data/trashnet_yolo

for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/trashnet_yolo/{split}/images')))
    print(f'  {split}: {imgs} images')

In [ ]:
# ── Cell 6：下載 TACO + 合併三層資料集 ────────────────────────────────────────
# ADR-002 Layer 1（主力）：含鋁箔包（Drink carton）標註
# 預計下載時間：20–40 分鐘（Flickr 圖片）
import os

# Step 1：Clone TACO repo（若已存在則跳過）
if not os.path.exists('data/TACO_repo'):
    !git clone https://github.com/pedropro/TACO.git data/TACO_repo
else:
    print('TACO_repo 已存在，跳過 clone')

# Step 2：確認 annotations.json 存在
ann_path = 'data/TACO_repo/data/annotations.json'
print('annotations.json 存在：', os.path.exists(ann_path))
print('TACO data/ 內容：', os.listdir('data/TACO_repo/data') if os.path.exists('data/TACO_repo/data') else 'NOT FOUND')

In [ ]:
# ── Cell 6b：從 Zenodo 下載完整 TACO（官方備份，不依賴 Flickr）──────────────
# Zenodo DOI: 10.5281/zenodo.3587843  共 2.7GB，下載約 5–10 分鐘
import os, glob

TACO_ZIP = 'data/TACO.zip'
TACO_OUT = 'data/TACO_zenodo'

if not os.path.exists(TACO_OUT):
    print('下載 TACO 完整資料集（2.7GB）...')
    !wget --show-progress "https://zenodo.org/record/3587843/files/TACO.zip" -O {TACO_ZIP}
    print('解壓中...')
    !unzip -q {TACO_ZIP} -d {TACO_OUT}
    print('解壓完成')
else:
    print('TACO_zenodo 已存在，跳過下載')

# 確認結構
ann_files = glob.glob(f'{TACO_OUT}/**/annotations.json', recursive=True)
print('annotations.json 位置：', ann_files)
img_count = len(glob.glob(f'{TACO_OUT}/**/*.jpg', recursive=True))
print(f'圖片總數：{img_count}')

In [ ]:
# ── Cell 6c：TACO 轉換 + 合併 → merged_v3 ─────────────────────────────────────
import os, glob

# 自動偵測 annotations.json（Zenodo 解壓路徑可能有一層子目錄）
ann_candidates = glob.glob('data/TACO_zenodo/**/annotations.json', recursive=True)
if not ann_candidates:
    raise FileNotFoundError('找不到 annotations.json，請確認 Cell 6b 已執行且下載成功')

taco_data_dir = os.path.dirname(ann_candidates[0])
print(f'TACO data 目錄：{taco_data_dir}')

!python data/prepare_taco.py \
    --taco-dir   "{taco_data_dir}" \
    --output-dir data/taco_yolo_v3

!python data/merge_datasets.py \
    --taco     data/taco_yolo_v3 \
    --trashnet data/trashnet_yolo \
    --output   data/merged_v3

print('\n合併後資料集 merged_v3：')
for split in ['train', 'val', 'test']:
    imgs = len(list(os.scandir(f'data/merged_v3/{split}/images')))
    print(f'  {split}: {imgs} images')

In [ ]:
# ── Cell 7：訓練 YOLO11m → waste_sorter_v3 ────────────────────────────────────
import os
from ultralytics import YOLO

# 優先使用含 TACO 鋁箔包的 v3 資料集
if os.path.exists('data/merged_v3/data.yaml'):
    DATA_YAML = 'data/merged_v3/data.yaml'
    print('使用合併資料集 v3（TACO Zenodo + TrashNet，含鋁箔包）')
elif os.path.exists('data/merged_v2/data.yaml'):
    DATA_YAML = 'data/merged_v2/data.yaml'
    print('使用合併資料集 v2')
elif os.path.exists('data/merged/data.yaml'):
    DATA_YAML = 'data/merged/data.yaml'
    print('使用合併資料集（TACO + TrashNet）')
else:
    DATA_YAML = 'data/trashnet_yolo/data.yaml'
    print('⚠️ 僅使用 TrashNet（Class 4 鋁箔包 = 0 張）')

model = YOLO('yolo11m.pt')

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=8,
    device=0,
    project='runs/train',
    name='waste_sorter_v3',
    exist_ok=True,
    patience=20,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
)

mAP = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f'\n訓練完成！mAP@50 = {mAP}')
print('模型路徑：runs/train/waste_sorter_v3/weights/best.pt')

In [ ]:
# ── Cell 8：下載 best_v3.pt ───────────────────────────────────────────────────
import glob, shutil, os
from google.colab import files

pts = glob.glob('**/best.pt', recursive=True)
print('找到的 best.pt：', pts)

# 優先取 v3，避免 SameFileError
src = next((p for p in pts if 'waste_sorter_v3' in p), None)
if src is None:
    src = next((p for p in pts if 'waste_sorter_v2' in p), None)
if src is None:
    src = next(p for p in pts if p != 'best_v3.pt')

print(f'使用模型：{src}')
shutil.copy(src, 'best_v3.pt')
print(f'Model size: {os.path.getsize("best_v3.pt")/1e6:.1f} MB')

files.download('best_v3.pt')
print('\n✅ 下載完成。接著在筆電執行：')
print('scp best_v3.pt jetson@<JETSON_IP>:~/AI-course/models/best.pt')